# Bài 5: Mô hình tốt đến đâu
**Mở hộp đen AI · Tập 1 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Bộ tin nhắn trong bài do nhóm tác giả tự soạn bằng cách ghép các cụm từ, không phải tin nhắn thật.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 1. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Bộ 200 tin nhắn: 170 tin thường, 30 tin rác
Ngoài đời tin rác ít hơn tin thường nhiều. Bộ dữ liệu này cố ý giữ sự **mất cân bằng** đó.

In [ ]:
import random

MO_RAC = ["chúc mừng bạn", "thông báo khẩn", "ưu đãi hôm nay",
          "tài khoản của bạn", "bạn có bưu phẩm", "mẹ ơi con đây",
          "em chào anh chị", "shop báo", "ngân hàng thông báo", "bạn ơi",
          "anh ơi"]
THAN_RAC = ["trúng thưởng xe máy", "được vay tiền lãi thấp",
            "bị khóa cần xác minh", "nhận quà miễn phí",
            "việc nhẹ lương cao tại nhà", "nạp thẻ nhận gấp đôi",
            "con đổi số mới chuyển tiền giúp con",
            "đơn hàng chưa giao cần xác nhận", "nhận tiền hoàn chưa",
            "mã xác nhận sắp hết hạn"]
KET_RAC = ["bấm link ngay", "liên hệ ngay", "trả lời tin này",
           "bấm vào link để nhận", "gọi số này nhé", "nhé", "ạ",
           "bấm link để xem"]

MO_THUONG = ["mẹ ơi", "con ơi", "bạn ơi", "cô thông báo", "lớp mình", "shop báo",
             "ngân hàng thông báo", "anh ơi", "thông báo"]
THAN_THUONG = ["mai kiểm tra toán", "tối nay học nhóm", "nhớ mang áo mưa",
               "đơn hàng đã giao", "chiều đá bóng", "con về muộn",
               "nhận tiền chưa", "mã xác nhận đã gửi",
               "họp phụ huynh sáng chủ nhật", "gửi mình bài tập với",
               "sinh nhật lan tối thứ bảy", "chuyển tiền học thêm rồi",
               "có lịch thi mới", "nhận quà sinh nhật chưa",
               "trúng thưởng bốc thăm của lớp rồi",
               "nhận quà miễn phí ở căng tin", "tài khoản học trực tuyến đã mở",
               "vay tiền mua sách được không"]
KET_THUONG = ["nhé", "nha", "không", "nhé con", "ạ", "bấm link để xem",
              "liên hệ cô nhé", "trả lời mẹ nhé", "liên hệ ngay",
              "trả lời tin này"]

def tao_tin(mo, than, ket, so_luong):
    tap = set()                    # dùng set để KHÔNG có tin trùng nhau
    while len(tap) < so_luong:
        tap.add(f"{random.choice(mo)} {random.choice(than)} {random.choice(ket)}")
    return sorted(tap)

random.seed(8)
rac = tao_tin(MO_RAC, THAN_RAC, KET_RAC, 30)
thuong = tao_tin(MO_THUONG, THAN_THUONG, KET_THUONG, 170)
tin = rac + thuong
nhan = ["rác"] * 30 + ["thường"] * 170
print("Tổng số tin:", len(tin), "| rác:", len(rac), "| thường:", len(thuong))
print("Ví dụ tin rác   :", rac[0])
print("Ví dụ tin thường:", thuong[0])

### Ô 2. Chia dữ liệu và thử một 'mô hình lười'
Mô hình lười không học gì cả: tin nào nó cũng đoán là 'thường'.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

tin_hoc, tin_thi, nhan_hoc, nhan_thi = train_test_split(
    tin, nhan, test_size=0.3, random_state=0, stratify=nhan)
print("Học:", len(tin_hoc), "tin | Thi:", len(tin_thi), "tin, trong đó",
      nhan_thi.count("rác"), "tin rác")

doan_luoi = ["thường"] * len(tin_thi)
diem_luoi = accuracy_score(nhan_thi, doan_luoi)
print("Độ chính xác của mô hình lười:", round(diem_luoi, 3))

### Ô 3. Huấn luyện bộ lọc Naive Bayes như ở Bài 3

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

dem = CountVectorizer()
X_hoc = dem.fit_transform(tin_hoc)
X_thi = dem.transform(tin_thi)            # phần thi: chỉ transform
mo_hinh = MultinomialNB().fit(X_hoc, nhan_hoc)
doan = mo_hinh.predict(X_thi)
print("Độ chính xác của Naive Bayes:", round(accuracy_score(nhan_thi, doan), 3))

### Ô 4. Ma trận nhầm lẫn: sai ở đâu, sai kiểu gì

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

mt = confusion_matrix(nhan_thi, doan, labels=["rác", "thường"])
print("                 đoán rác   đoán thường")
print(f"thật là rác      {mt[0, 0]:8}   {mt[0, 1]:11}")
print(f"thật là thường   {mt[1, 0]:8}   {mt[1, 1]:11}")
print()
pre = precision_score(nhan_thi, doan, pos_label="rác")
rec = recall_score(nhan_thi, doan, pos_label="rác")
print("Precision (rác):", round(pre, 3))
print("Recall    (rác):", round(rec, 3))

### Ô 5. Xem tận mắt những tin máy đoán sai

In [ ]:
for t, that, d in zip(tin_thi, nhan_thi, doan):
    if that != d:
        print(f"thật: {that:6} | đoán: {d:6} | {t}")

## Chặng 2: Đọc
Trả lời vào vở trước khi đi tiếp:
1. Mô hình lười không học gì mà vẫn đạt 85%. Vì sao? Nó bắt được bao nhiêu tin rác?
2. Trong ma trận ở ô 4, ô nào là 'để lọt', ô nào là 'chặn nhầm'?
3. Đọc các tin sai ở ô 5. Theo em vì sao máy nhầm những tin này?

## Chặng 3: Sửa
### Ô 6. Vặn ngưỡng: khó tính hay dễ tính?
Mặc định, máy gọi một tin là rác khi nó chắc từ 50% trở lên. Thử đổi ngưỡng đó.

In [ ]:
p_rac = mo_hinh.predict_proba(X_thi)[:, 0]        # cột 0 là xác suất 'rác'
print("ngưỡng | bắt đúng | chặn nhầm | để lọt | precision | recall")
for nguong in [0.1, 0.3, 0.5, 0.7, 0.9]:
    d = ["rác" if p >= nguong else "thường" for p in p_rac]
    m = confusion_matrix(nhan_thi, d, labels=["rác", "thường"])
    pre = m[0, 0] / max(m[0, 0] + m[1, 0], 1)
    rec = m[0, 0] / (m[0, 0] + m[0, 1])
    print(f"  {nguong:.1f}  | {m[0, 0]:8} | {m[1, 0]:9} | {m[0, 1]:6}"
          f" | {pre:9.2f} | {rec:6.2f}")

### Ô 7. Học vẹt: thuộc lòng đề cũ, gặp đề mới thì chịu
Mô hình 'học vẹt' ghi nhớ nguyên văn từng tin đã học. Gặp tin lạ, nó đoán bừa là 'thường'.

In [ ]:
so_tay = dict(zip(tin_hoc, nhan_hoc))             # thuộc lòng: tin -> nhãn

def hoc_vet(ds_tin):
    return [so_tay.get(t, "thường") for t in ds_tin]

diem_hoc = accuracy_score(nhan_hoc, hoc_vet(tin_hoc))
diem_thi = accuracy_score(nhan_thi, hoc_vet(tin_thi))
print(f"Học vẹt     | bài đã học: {diem_hoc:.3f} | đề thi: {diem_thi:.3f}")
print(f"Naive Bayes | bài đã học: {mo_hinh.score(X_hoc, nhan_hoc):.3f}"
      f" | đề thi: {mo_hinh.score(X_thi, nhan_thi):.3f}")

### Ô 8. Cây quyết định: càng sâu càng thuộc lòng

In [ ]:
from sklearn.tree import DecisionTreeClassifier

print("độ sâu | điểm trên bài đã học | điểm trên đề thi")
for sau in [1, 2, 3, 5, 10, None]:
    cay = DecisionTreeClassifier(max_depth=sau, random_state=0)
    cay.fit(X_hoc, nhan_hoc)
    hoc, thi = cay.score(X_hoc, nhan_hoc), cay.score(X_thi, nhan_thi)
    print(f"{str(sau):>6} | {hoc:20.3f} | {thi:16.3f}")

## Chặng 4: Tự làm
1. Ở ô 1, đổi `random.seed(8)` thành một số khác rồi chạy lại toàn bộ. Các con số ở ô 3 và ô 4 thay đổi bao nhiêu? Điều đó nói gì về độ tin cậy của một lần đo?
2. Em là nhà mạng. Em chọn ngưỡng nào ở ô 6? Viết hai câu bảo vệ lựa chọn, có nhắc tới 'chặn nhầm' và 'để lọt'.
3. Thêm 10 tin rác do em tự viết vào danh sách `rac` (giữ nguyên phần còn lại) và xem recall thay đổi thế nào.

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*